In [1]:
from pyspark.sql.functions import dayofmonth, month, year, to_date, substring, when, regexp_replace, col
bronze_table_path_results = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Results_Bronze.Lakehouse/Tables/dbo/results"
bronze_table_path_former_names = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Results_Bronze.Lakehouse/Tables/dbo/former_names"
bronze_table_path_goalscorers = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Results_Bronze.Lakehouse/Tables/dbo/goalscorers"
bronze_table_path_shootouts = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Results_Bronze.Lakehouse/Tables/dbo/shootouts"

df_results=spark.read.format("delta").load(bronze_table_path_results)
df_former_names =spark.read.format("delta").load(bronze_table_path_former_names)
df_goalscorers=spark.read.format("delta").load(bronze_table_path_goalscorers)
df_shootouts=spark.read.format("delta").load(bronze_table_path_shootouts)

StatementMeta(, ba987f43-eaf7-42ef-b6d0-6196561859a6, 3, Finished, Available, Finished, False)

In [2]:
df_results_transformed= df_results.withColumn("day", dayofmonth(col("date"))).withColumn("month", month(col("date"))).withColumn("year", year(col("date")))
#df_former_names_transformed= df_former_names.withColumn("day", dayofmonth(col("date"))).withColumn("month", month(col("date"))).withColumn("year", year(col("date")))
df_goalscorers_transformed = df_goalscorers \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("half", 
        when(col("minute") <= 45, "1st half")
        .when((col("minute") > 45) & (col("minute") <= 90), "2nd half")
        .when((col("minute") > 90) & (col("minute") <= 105), "ET1")
        .when(col("minute") > 105, "ET2")
    )
df_shootouts_transformed= df_shootouts.withColumn("day", dayofmonth(col("date"))).withColumn("month", month(col("date"))).withColumn("year", year(col("date")))


StatementMeta(, ba987f43-eaf7-42ef-b6d0-6196561859a6, 4, Finished, Available, Finished, False)

In [3]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW results_golscorers AS
SELECT
    r.date,
    r.home_team,
    r.away_team,
    r.home_score,
    r.away_score,
    r.tournament,
    r.city,
    r.country,
    r.neutral,
    g.team,
    g.scorer,
    g.minute,
    g.own_goal,
    g.penalty
FROM Football.LH_Results_Bronze.dbo.results r
LEFT JOIN Football.LH_Results_Bronze.dbo.goalscorers g
    ON r.date = g.date
    AND r.home_team = g.home_team
    AND r.away_team = g.away_team

StatementMeta(, ba987f43-eaf7-42ef-b6d0-6196561859a6, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
%%sql
SELECT * from confederations

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.

In [4]:
silver_table_path_results = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/dbo/results"
silver_table_path_former_names = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/dbo/former_names"
silver_table_path_goalscorers = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/dbo/goalscorers"
silver_table_path_shootouts = "abfss://Football@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/dbo/shootouts"

StatementMeta(, ba987f43-eaf7-42ef-b6d0-6196561859a6, 6, Finished, Available, Finished, False)

In [5]:
df_results_transformed.write.format("delta").mode("overwrite").save(silver_table_path_results)
df_goalscorers_transformed.write.format("delta").mode("overwrite").save(silver_table_path_goalscorers)
df_shootouts_transformed.write.format("delta").mode("overwrite").save(silver_table_path_shootouts)
df_former_names.write.format("delta").mode("overwrite").save(silver_table_path_former_names)

StatementMeta(, ba987f43-eaf7-42ef-b6d0-6196561859a6, 7, Finished, Available, Finished, False)